In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"
OUTPUT_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

GLOMERULUS_OUTPUT = os.path.join(
    OUTPUT_FOLDER,
    "gbm_glomerulus_summary.csv"
)

PATIENT_OUTPUT = os.path.join(
    OUTPUT_FOLDER,
    "gbm_patient_summary.csv"
)

GLOBAL_PLOT = os.path.join(
    OUTPUT_FOLDER,
    "global_glomerulus_boxplot.png"
)

PATIENT_PLOT = os.path.join(
    OUTPUT_FOLDER,
    "patient_glomerulus_boxplot.png"
)

if not os.path.exists(INPUT_CSV):

    raise FileNotFoundError(
        f"Input file not found:\n{INPUT_CSV}"
    )

df = pd.read_csv(INPUT_CSV)

required_columns = [
    "patient_id",
    "glomerulus_id",
    "component_id",
    "median_thickness_nm"
]

missing = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing:

    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(missing)
    )

print()
print("=" * 70)
print("GBM GLOMERULUS-LEVEL STATISTICAL ANALYSIS")
print("=" * 70)

print(
    f"Membrane components : {len(df)}"
)

print(
    f"Patients             : "
    f"{df['patient_id'].nunique()}"
)

print(
    f"Glomeruli             : "
    f"{df[['patient_id', 'glomerulus_id']].drop_duplicates().shape[0]}"
)

df["unique_glomerulus_id"] = (
    df["patient_id"].astype(str)
    + "_"
    + df["glomerulus_id"].astype(str)
)

glomerulus_summary = (

    df.groupby(
        [
            "patient_id",
            "glomerulus_id",
            "unique_glomerulus_id"
        ]
    )
    .agg(

        number_of_membrane_components=(
            "component_id",
            "count"
        ),

        mean_thickness_nm=(
            "median_thickness_nm",
            "mean"
        ),

        median_thickness_nm=(
            "median_thickness_nm",
            "median"
        ),

        std_thickness_nm=(
            "median_thickness_nm",
            "std"
        ),

        min_thickness_nm=(
            "median_thickness_nm",
            "min"
        ),

        max_thickness_nm=(
            "median_thickness_nm",
            "max"
        )

    )

    .reset_index()
)

glomerulus_summary["std_thickness_nm"] = (
    glomerulus_summary["std_thickness_nm"]
    .fillna(0)
)

glomerulus_summary.to_csv(
    GLOMERULUS_OUTPUT,
    index=False
)

print()
print("Glomerulus-level summary saved:")
print(GLOMERULUS_OUTPUT)

print()
print(
    "Total glomeruli analysed :",
    len(glomerulus_summary)
)

values = (
    glomerulus_summary[
        "median_thickness_nm"
    ]
    .dropna()
)

print()
print("GLOBAL GLOMERULUS STATISTICS")
print("=" * 60)

print(
    f"Number of glomeruli : {len(values)}"
)

print(
    f"Mean thickness      : {values.mean():.2f} nm"
)

print(
    f"Median thickness    : {values.median():.2f} nm"
)

print(
    f"Standard deviation  : {values.std():.2f} nm"
)

print(
    f"Minimum             : {values.min():.2f} nm"
)

print(
    f"Maximum             : {values.max():.2f} nm"
)

patient_summary = (

    glomerulus_summary

    .groupby("patient_id")

    .agg(

        number_of_glomeruli=(
            "unique_glomerulus_id",
            "count"
        ),

        mean_thickness_nm=(
            "median_thickness_nm",
            "mean"
        ),

        median_thickness_nm=(
            "median_thickness_nm",
            "median"
        ),

        std_thickness_nm=(
            "median_thickness_nm",
            "std"
        ),

        min_thickness_nm=(
            "median_thickness_nm",
            "min"
        ),

        max_thickness_nm=(
            "median_thickness_nm",
            "max"
        )

    )

    .reset_index()
)

patient_summary["std_thickness_nm"] = (
    patient_summary["std_thickness_nm"]
    .fillna(0)
)

patient_summary.to_csv(
    PATIENT_OUTPUT,
    index=False
)

print()
print("Patient-level summary saved:")
print(PATIENT_OUTPUT)

print()
print("PATIENT-LEVEL STATISTICS")
print("=" * 60)

print(
    patient_summary.to_string(
        index=False
    )
)

fig, ax = plt.subplots(
    figsize=(9, 7)
)

ax.boxplot(
    values,
    showfliers=True
)

# Add individual glomerulus points
x_positions = np.random.normal(
    1,
    0.04,
    size=len(values)
)

ax.scatter(
    x_positions,
    values,
    alpha=0.65,
    s=35
)

ax.set_ylabel(
    "GBM thickness (nm)",
    fontsize=12
)

ax.set_title(
    "Distribution of GBM Thickness Across Glomeruli",
    fontsize=14,
    fontweight="bold"
)

ax.set_xticks([1])

ax.set_xticklabels(
    ["All glomeruli"]
)

ax.grid(
    axis="y",
    alpha=0.2,
    linestyle="--"
)

explanation = (
    "How to read this plot\n"
    "• Each point = one glomerulus\n"
    "• Box = middle 50% of glomeruli (IQR)\n"
    "• Centre line = median\n"
    "• Whiskers = 1.5 × IQR\n"
    "• Points beyond whiskers = potential outliers"
)

ax.text(
    1.02,
    0.98,
    explanation,
    transform=ax.transAxes,
    fontsize=9,
    verticalalignment="top",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="gray",
        alpha=0.9
    )
)

plt.tight_layout()

plt.savefig(
    GLOBAL_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print()
print("Global glomerulus box plot saved:")
print(GLOBAL_PLOT)

patients = sorted(
    glomerulus_summary[
        "patient_id"
    ].unique()
)
data = []

for patient in patients:

    patient_values = (

        glomerulus_summary[
            glomerulus_summary["patient_id"] == patient
        ]

        ["median_thickness_nm"]

        .dropna()
        .values

    )

    data.append(
        patient_values
    )


fig, ax = plt.subplots(
    figsize=(13, 7)
)

ax.boxplot(
    data,
    tick_labels=patients,
    showfliers=True
)

# Add individual glomerulus points
for i, patient_values in enumerate(
    data,
    start=1
):

    x_positions = np.random.normal(
        i,
        0.04,
        size=len(patient_values)
    )

    ax.scatter(
        x_positions,
        patient_values,
        alpha=0.65,
        s=25
    )

ax.set_xlabel(
    "Patient",
    fontsize=12
)

ax.set_ylabel(
    "GBM thickness (nm)",
    fontsize=12
)

ax.set_title(
    "GBM Thickness Distribution by Patient",
    fontsize=14,
    fontweight="bold"
)

ax.grid(
    axis="y",
    alpha=0.2,
    linestyle="--"
)

explanation = (
    "How to read this plot\n"
    "• Each point = one glomerulus\n"
    "• Box = middle 50% of glomeruli (IQR)\n"
    "• Centre line = median\n"
    "• Whiskers = 1.5 × IQR\n"
    "• Points beyond whiskers = potential outliers"
)

ax.text(
    1.02,
    0.98,
    explanation,
    transform=ax.transAxes,
    fontsize=9,
    verticalalignment="top",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="gray",
        alpha=0.9
    )
)

plt.tight_layout()

plt.savefig(
    PATIENT_PLOT,
    dpi=300,
    bbox_inches="tight"
)
plt.close()

print()
print("Patient box plot saved:")
print(PATIENT_PLOT)

print()
print("=" * 70)
print("GLOMERULUS-LEVEL ANALYSIS COMPLETED")
print("=" * 70)

print()
print("Generated files:")

print(
    "1.",
    GLOMERULUS_OUTPUT
)

print(
    "2.",
    PATIENT_OUTPUT
)

print(
    "3.",
    GLOBAL_PLOT
)

print(
    "4.",
    PATIENT_PLOT
)


GBM GLOMERULUS-LEVEL STATISTICAL ANALYSIS
Membrane components : 394
Patients             : 11
Glomeruli             : 257

Glomerulus-level summary saved:
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\gbm_glomerulus_summary.csv

Total glomeruli analysed : 257

GLOBAL GLOMERULUS STATISTICS
Number of glomeruli : 257
Mean thickness      : 229.73 nm
Median thickness    : 190.10 nm
Standard deviation  : 143.70 nm
Minimum             : 82.37 nm
Maximum             : 1340.00 nm

Patient-level summary saved:
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\gbm_patient_summary.csv

PATIENT-LEVEL STATISTICS
patient_id  number_of_glomeruli  mean_thickness_nm  median_thickness_nm  std_thickness_nm  min_thickness_nm  max_thickness_nm
     01-24                   15         204.593703           174.481620         97.825877        113.405849        480.615405
     02-24                   29         309.660627           241.313279        235.371648        107.744507       1340.001960